# pw01 — Sionna 로 ISAC 센싱을 한 선행 논문들

> ⚠ **이 노트북은 생성물이다. 수정은 `prior_work/src/make_pw01.py` 에서** 하고 재실행할 것.
> 모든 사실·인용은 `prior_work/outputs/prior_work.json`(2× 딥리서치 + 직접 웹확인) 에서 주입한다.

**두 질문에 답한다:** ① *Sionna 로 ISAC(센싱)을 한 선행 논문이 실제로 존재하나?* 
② 존재한다면 *우리가 마주한 간극(스톡 Sionna 는 소형 표적의 코히어런트 RCS 를 메쉬에서 못 낸다, report06)을 그들은 어떻게 우회했나?*

**검증 등급(정직성 장치):** ✅검증=CONFIRMED · 🔎직접확인=WEB · 📄단일출처=SOURCE · ⚠미검증단서=IMAGE_LEAD
— 다른 AI 답변에 나온 논문명은 환각 가능성이 있어, **실재를 1차 출처로 확인한 것만** 싣는다.

## §1. 한 줄 답

**① 존재한다 — 다수.** 아래 8건은 전부 실재를 확인했다(arXiv/GitHub 1차 출처). 
**② 소형 표적 코히어런트 RCS 를 스톡 Sionna 로 푼 선행은 없다.** 표준 우회는 세 갈래다:

| 우회 | 뜻 | 대표 선행 |
|---|---|---|
| **(b) 확산 산란계수 S** | 표면에 S∈[0,1] 를 주고 산란전력을 S² 로 배분(재질별, 실측 보정) | Great-X, Deterministic-Modeling |
| **(c) RCS 를 별도 계산/가정해 주입** | σ(상수 또는 자세의존 LUT)를 채널에 넣음 (h = h_bg + h_target) | **LAMBDA**(CADFEKO), **Temporal-GNN**(점산란체), NIST 5GNRad·3GPP |
| **(d) 커스텀 산란 add-on** | Sionna 를 기저엔진으로만 쓰고 산란모델을 직접 얹음 | Ziganshin(UTD 회절), **우리(SBR+PO)** |

⭐ **우리 방식(자작 RCS 계산 → Sionna 채널 주입)은 (d)+(c) 조합인데, 이게 바로 최신 Sionna-ISAC 선도 연구가 쓰는 방식이다** — LAMBDA 는 CADFEKO 로, Temporal-GNN 은 점산란체로 RCS 를 별도 계산해 Sionna 채널에 넣는다. 우리는 상용 CADFEKO 대신 자작 SBR+PO 를 쓰고, 능동 FMCW 대신 패시브 바이스태틱이라는 점이 다르다(자세히 §2·§4·pw03).

---
## §1b. ⭐ 한눈에 보는 표 — Sionna 를 센싱에 쓴 연구는 어떻게 결합했나

질문(사용자 요청): *Sionna 로 센싱 시뮬레이션을 한 연구들은 Sionna 를 **어떻게 활용**했고, **어떤 라이브러리**와 **어떤 방식**으로 결합했나?* — 팀미팅 문헌표 형식으로 정리한다. **맨 아래 굵은 줄이 우리 위치**다.

| 연도 | 연구(플랫폼) | 게재처(신뢰성) | 센싱 대상 | Sionna 활용(무엇을) | 결합 도구/라이브러리 | 결합 방식 | 표적 산란 |
|---|---|---|---|---|---|---|---|
| 2026 | LAMBDA (데이터셋) | arXiv:2607.03826 (preprint·2.04TB) | 저고도 UAV | 재질인식 RT: multipath CSI·지연·도플러·각도 | CADFEKO + UE5·AirSim·Blender | Sionna CSI × CADFEKO 자세의존 RCS × virtual-array 조향 → FMCW radar cube | (c) 외부 RCS 주입 |
| 2026 | Temporal-GNN for ISAC | arXiv:2604.08306 (preprint) | 이동 표적 | 바이스태틱 ISAC CIR 생성 | 자체 점산란체 RCS·안테나패턴 | CIR 에 외부 점산란체 bistatic RCS 주입 → delay-Doppler → GNN 추적 | (c) 점산란체 주입 |
| 2025 | Great-X (Unreal is all you need) | arXiv:2507.08716 (preprint) | 저고도 UAV | **Sionna식 RT 를 Unreal 안에 재구현** | Unreal Engine | 단일엔진에서 RT 채널 모델링(확산계수 S) | (b) 확산계수 S |
| 2026 | Deterministic Modeling ISAC | arXiv:2603.28736 (**EuCAP 2026, peer-reviewed**) | 차량(모노·바이스태틱) | RT 내장 확산 산란 | (측정 79GHz 채널사운딩) | 재질별 S 를 실측 보정(R²+S²=1) | (b) 확산계수 S |
| 2026 | Ziganshin (Discretized Curved Bodies) | arXiv:2604.05991 (preprint·TU Ilmenau) | 구·원통·차량 | Sionna-RT v0.19 를 **기저 엔진**으로만 | 자체 UTD 회절 확장 | 경면 facet + UTD 회절 add-on | (d) 커스텀 산란 |
| 2025 | CISSIR (NVIDIA 공식) | arXiv:2502.10371 (**NVIDIA Made-with-Sionna**) | (물리표적 없음) | 빔 코드북·자기간섭 저감 | — | 센싱 성능=SI 저감·양자화한계 | — (표적 RCS 없음) |
| 2026 | **우리(sionna2)** | (진행) | DJI 드론 5종 | 챔버 전파·직접파·바닥유령·지연커널·9모드 파형 | **자작 SBR+PO**(복소장 E) · pyAPRiL(검출·검증) · OpenISAC(실측) | Sionna 채널에 SBR+PO σ(복소) 주입 h=h_dir+h_bg+h_target → pyAPRiL ECA/CAF/CFAR | (d)+(c) 자작 SBR+PO 계산→주입 |

> 우리와 같은 열(Sionna 전파 + 외부 RCS 주입)에 LAMBDA·Temporal-GNN 이 있고, 우리는 상용 CADFEKO 대신 자작 SBR+PO(복소장), 능동 FMCW 대신 패시브 바이스태틱 OFDM. 표적산란 (b)확산S/(c)점산란체/(d)커스텀 중 우리는 (d)로 계산해 (c)로 주입 = 3GPP Rel-19 표준(h_bg+h_target)과 정합.

**표적 산란 처리 3분류**(어느 칸이든 이 중 하나): **(b)** 확산계수 S 가정 · **(c)** 외부 RCS 를 별도 계산/가정해 채널에 주입 · **(d)** 커스텀 산란 add-on. Sionna 자체는 소형 표적의 코히어런트 RCS 를 메쉬에서 못 내므로(→§5), 센싱 연구는 전부 이 셋 중 하나로 **RCS 를 밖에서 넣는다**.

---
## §1c. 관련 선례 표 — 비-Sionna 이지만 우리 태스크에 직접 맞닿은 연구

Sionna 를 안 썼어도 **패시브 바이스태틱·드론 RCS/마이크로도플러·5G 조명원·3GPP 표준**에서 우리와 직접 겹치는 선례들. (드론 문헌 목록 자체는 팀미팅 덱 LTE/5G/WiFi 패시브 레이더 표 참고.)

| 연도 | 연구 | 게재처(신뢰성) | 조명원/구조 | 방법 | 표적/드론 | 우리와의 관계 |
|---|---|---|---|---|---|---|
| 2025 | Costa & Thomä (multi-propeller μD) | **IEEE J-STEAP 2025(peer-reviewed)**·arXiv:2504.05168 | 분산 ISAC·바이스태틱 OFDM | thin-wire 점산란체+PO 로터 RCS+정적동체, 측정검증 | 다중 프로펠러 드론 | ⭐드론 멀티산란체 바이스태틱 모델의 최근접 선례(우리 SBR+PO 대응) |
| 2024 | Costa & Thomä (RadarConf24) | IEEE RadarConf24·arXiv:2401.14287 | 분산 ISAC | 프로펠러 μD 수학모델, 측정검증 | 드론 프로펠러 | 위 J-STEAP 의 학회판 |
| 2026 | Wypich & Zielinski | **Sensors 2026(peer-reviewed)**·AGH+Ericsson | 5G NR 패시브 바이스태틱 | **USRP X310, ECA+ → CA-CFAR**, PDSCH 사용자데이터 활용 | 차량(실측 OTA) | ⭐우리 ECA→CFAR 실하드웨어 선례; '기준=풀웨이브폼' 실증(POD 24→78%) |
| 2025 | Jopanya & Osorio | **IEEE SPAWC 2025(accepted)**·arXiv:2504.02641 | 5G NR **SSB** 패시브 바이스태틱 | 점표적 CRB(거리·속도) | 저고도 드론 | 최근접 패시브-5G SSB-드론(우리 G1 최악모드) |
| 2026 | 3GPP Rel-19 ISAC Sim (Putirf) | arXiv:2606.07328 + **오픈 MATLAB** | 3GPP TR38.901 채널 | target/background 채널·점산란체 RCS | (표적 일반) | h=h_bg+h_target 표준 구현 — 우리 주입구조의 표준 근거·교차검증 후보 |
| 2025 | Zhang et al.(BUPT) Unified RCS | arXiv:2505.20673 (BUPT 3GPP 주도) | 3GPP RCS 표준 | **측정 UAV RCS 5밴드**(전력×각도×랜덤) | UAV·인체·차량 | 측정 UAV RCS 앵커 + 구조적 RCS 주입이 우리와 정합 |
| 2025 | NYU/Rappaport ISAC Imaging | arXiv:2509.06672 (NYU WIRELESS) | NYURay(비-Sionna) 바이스태틱 | CSI 경면 멀티패스→3D 반사점 영상화 | 물체 표면 | 기하 광선추적은 경면만—RCS 크기는 외부공급 필요(우리 주장 재확인) |

> 🔑 **핵심.** 우리 접근의 세 조각이 각각 강한 선례를 갖는다 — 드론 멀티산란체 바이스태틱 μD(**Costa & Thomä, IEEE J-STEAP peer-reviewed**), ECA→CFAR 5G 패시브 레이더 실측(**Wypich & Zielinski, Sensors, USRP X310**), 외부 RCS 주입 h=h_bg+h_target(**3GPP Rel-19 표준 오픈 구현**). 우리 기여는 이 셋의 **결합**(패시브 바이스태틱 + 자작 SBR+PO 드론 RCS + 상시vs세션 9모드)이다.

---
## §2. ⭐ 가장 직접적인 선례 — 'Sionna + 외부 RCS 주입' 하이브리드

### 🔎직접확인  LAMBDA: A Low-Altitude Multimodal Base Dataset for UAV Sensing and Communication
- **저자·출처**: (연구진 미확인 — arXiv 확인 필요) — arXiv:2607.03826 (2026-07)  ([1](https://arxiv.org/pdf/2607.03826) · [2](https://arxiv.org/abs/2607.03826))
- **신뢰성(연구진·게재처)**: arXiv preprint(2026-07) · **2.04 TB·517,939 프레임 대규모 데이터셋**(상당한 공수) · 게재 예정 venue 미확인(preprint) — 규모는 크나 peer-review 상태 미확정, 참고 시 가중 유의
- **무엇을 센싱?** 소형 UAV(드론) — 도시/교외/캠퍼스, 다중UAV·다중BS, 야간·기상(맑음/비/눈/안개)
- **표적 산란 처리**: ⭐ **c) 외부 RCS 주입 하이브리드** — Sionna RT(재질인식 광선추적: multipath CSI·지연·도플러·각도) + **CADFEKO(상용 EM)로 UAV 자세의존 RCS** 를 별도 계산해 결합. radar synthesis 가 Sionna 복소 CSI 경로계수 × 왕복지연 × 도플러 × virtual-array 조향위상 × **UAV 자세의존 RCS** × 수신잡음
- **검출체인?** FMCW radar cube → Range-Doppler·Range-Angle (능동 FMCW, 77 GHz)
- **우리와의 관계**: ⭐⭐ **우리 하이브리드의 가장 직접적 선례** — 'Sionna multipath + 외부 UAV RCS'가 정확히 우리 구조다. 차이: LAMBDA=**상용 CADFEKO** RCS·**능동 FMCW 77GHz**·프로펠러 μD 미정밀 / 우리=**자작 SBR+PO** RCS(복소장까지)·**패시브 바이스태틱 WiFi/LTE/5G**·프로펠러 μD. → 우리 접근이 이상하지 않음의 결정적 근거

### 🔎직접확인  Temporal Graph Neural Network for ISAC Target Detection and Tracking
- **저자·출처**: (arXiv) — arXiv:2604.08306 (2026-04)  ([1](https://arxiv.org/abs/2604.08306))
- **신뢰성(연구진·게재처)**: arXiv preprint(2026-04) — peer-review 상태 미확정, 방법론 참고용
- **무엇을 센싱?** 이동 표적(다중), delay-Doppler 맵 기반
- **표적 산란 처리**: ⭐ **c) 외부 점산란체 바이스태틱 RCS 주입** — Sionna 광선추적으로 CIR 생성, RCS 는 안테나패턴·경로손실·표적RCS 포함하되 **표적은 점산란체 가정**(bistatic RCS externally prescribed)
- **검출체인?** delay-Doppler 맵 → TGNN 추적(node classification·데이터연관), Kalman 대비 NMSE↓
- **우리와의 관계**: 'Sionna 전파 + 외부 point-target RCS 주입'이 **검출/추적 알고리즘 연구의 표준 관행**임을 보임 (우리 h=h_bg+h_target 와 동일). 추적에 GNN 을 쓴 점은 우리 future work 후보

> 🔑 **이 둘이 왜 결정적인가.** "σ 를 따로 구해 주입하는 게 편법 아닌가?" 라는 의문의 답이다 — **아니다, 그게 선도 방식이다.** LAMBDA(대규모 데이터셋)는 Sionna 전파에 **CADFEKO 로 계산한 UAV RCS** 를 결합하고, Temporal-GNN(추적 연구)은 **외부 점산란체 RCS** 를 주입한다. 우리는 그 '외부 RCS 계산기'를 상용툴 대신 **자작 SBR+PO** 로 두었을 뿐, 구조는 동일하다. ⚠ 단 둘 다 arXiv preprint 라 게재 확정은 아니다(신뢰성 항목 참고).

---
## §3. Sionna 로 드론/차량을 센싱한 선행 (표적 산란 처리 비교)

### ✅검증  Unreal is all you need: Multimodal ISAC Data Simulation with Only One Engine
- **저자·출처**: Kongwu Huang, Shiyi Mu, Jun Jiang, Yuan Gao (Shanghai Univ) · Shugong Xu (XJTLU) — arXiv:2507.08716 (2025-07, cs.CV) · 플랫폼명 Great-X  ([1](https://arxiv.org/abs/2507.08716) · [2](https://arxiv.org/html/2507.08716v3))
- **신뢰성(연구진·게재처)**: arXiv preprint(2025-07) · Shanghai Univ·XJTLU · 데이터셋/플랫폼 논문(대규모)
- **무엇을 센싱?** 소형 저고도 UAV(드론) — 상하이 루자쭈이 상공 비행, Great-MSD 데이터셋(CSI+RGB+Radar+LiDAR)
- **표적 산란 처리**: b) 확산 산란계수 S∈[0,1] (S² 확산 + 경면). Sionna 식 광선추적을 Unreal 안에 재구현, Sionna RT 로 교차검증
- **검출체인?** 없음 — CFAR/거리-도플러 아님. CSI 기반 UAV 3D 위치추정
- **우리와의 관계**: 드론 RCS 를 메쉬에서 코히어런트 계산하지 않음(확산계수 사용) → 우리 주장과 모순 없음. 드론을 '산란 표적'이 아니라 '통신/센싱 플랫폼'으로 다룸

### 📄단일출처  Deterministic Modeling of Dynamic ISAC Channels in RF Digital Twin Environments
- **저자·출처**: Cesar Montaner, Saúl Fenollosa, Andres Ortega, Hugo Beltrán, Narcis Cardona (iTEAM, U. Politècnica de València) — arXiv:2603.28736 · EuCAP 2026 채택  ([1](https://arxiv.org/abs/2603.28736))
- **신뢰성(연구진·게재처)**: ⭐ **EuCAP 2026 채택(peer-reviewed)** · iTEAM UPV(Narcis Cardona — 채널모델 저명그룹) — 신뢰도 상
- **무엇을 센싱?** 실차량 — Nissan Micra(모노스태틱), KIA Xceed(바이스태틱, UE 탑재). 드론/소형 표적 없음
- **표적 산란 처리**: b) Sionna 내장 확산 산란(R²+S²=1), 재질별 S 를 79GHz E-band 채널 사운딩 실측으로 보정
- **검출체인?** 채널 레벨(디지털 트윈 채널 생성)
- **우리와의 관계**: ⭐ **우리 주장 직접 지지** — 논문 스스로 '메쉬 위 순수 경면 광선추적은 mmWave 산란전력을 과소평가하고, 정확한 EM 산란에 필요한 λ/10 메싱은 계산상 불가능'이라고 명시. = report06 의 핵심 주장

### 📄단일출처  Ray-Based Simulation of Scattering from Discretized Curved Bodies for Vehicular and ISAC Applications
- **저자·출처**: Ainur Ziganshin, Enrico M. Vitucci, Wim Kotterman, Reiner Thomä, Christian Schneider, Vittorio Degli-Esposti — arXiv:2604.05991 (eess.SP, 2026-04)  ([1](https://arxiv.org/pdf/2604.05991))
- **신뢰성(연구진·게재처)**: arXiv + EuCAP 계열 · Degli-Esposti·Vitucci(Bologna, 전파전파 저명그룹) — 신뢰도 상
- **무엇을 센싱?** 정준 곡면체(구·반경 7λ 원통) + 실차량 i-MiEV(1496/220 facet) @2-3GHz. 소형/서브미터 표적 없음
- **표적 산란 처리**: d) **커스텀 add-on** — Sionna-RT v0.19 를 기저 엔진으로만 쓰고, 저자들이 UTD 회절(엣지·꼭짓점·이중엣지) 확장을 직접 구현. 경면 facet + UTD. **PO/SBR 은 관련연구로 인용만 하고 안 씀**
- **검출체인?** 해석해·FEKO MLFMM·BIRA 실측 대조 검증
- **우리와의 관계**: 'Sionna 에 커스텀 산란을 더한다'는 우리와 같은 정신. 단 방법은 UTD(우리는 PO/SBR), 표적은 대형

> 🔑 **읽는 법.** 셋 다 '표적을 메쉬에서 코히어런트 RCS 로 계산'하지 **않는다**. Great-X·Deterministic-Modeling 은 **확산계수 S**(b)로, Ziganshin 은 **커스텀 UTD**(d)로 우회한다. 우리와 정신이 가장 가까운 건 Ziganshin(‘Sionna + 직접 만든 산란’)이지만, 그들은 UTD·대형표적, 우리는 PO/SBR·소형 드론이다.

---
## §4. Sionna 를 ISAC 에 쓰지만 '표적 RCS'는 안 하는 선행 (문맥용)

### 📄단일출처  CISSIR: Beam Codebooks with Self-Interference Reduction Guarantees for ISAC Beyond 5G
- **저자·출처**: Rodrigo Hernangómez, Jochen Fink, Renato L. G. Cavalcante, Sławomir Stańczak — arXiv:2502.10371 (2025) · **NVIDIA 공식 'Made with Sionna' 유일 ISAC 등재** · Sionna v0.17 · code github.com/rodrihgh/cissir  ([1](https://nvlabs.github.io/sionna/made_with_sionna.html) · [2](https://arxiv.org/abs/2502.10371))
- **신뢰성(연구진·게재처)**: ⭐ **NVIDIA 공식 Made-with-Sionna 등재** · Fraunhofer HHI(Stańczak) — 신뢰도 상
- **무엇을 센싱?** 물리 표적 없음 — 자기간섭 저감·양자화잡음 한계로 센싱 성능만 다룸
- **표적 산란 처리**: n/a
- **검출체인?** n/a
- **우리와의 관계**: 스톡 Sionna 를 표적 RCS 에 쓰지 않음 — 우리 주장과 정합. NVIDIA 공식 쇼케이스에도 소형표적 RCS ISAC 이 없다는 방증

### ✅검증  SimART: A Unified and Open Real-world Multimodal Simulation Platform for 6G ISAC
- **저자·출처**: Kang Yan, Yuqi Cao, Jiaqi Li, Luping Xiang, Kun Yang (UESTC, Nanjing Univ) — arXiv:2605.13309 (2026-05) · github.com/guchuanv-alt/SimART (~168★)  ([1](https://arxiv.org/abs/2605.13309) · [2](https://github.com/guchuanv-alt/SimART))
- **신뢰성(연구진·게재처)**: arXiv preprint(2026-05) · UESTC·Nanjing · GitHub 168★ — 신뢰도 중
- **무엇을 센싱?** UAV 를 **비전(YOLOv8 on RGB)+GPS** 로 센싱(빔예측용). 레이더 에코 아님
- **표적 산란 처리**: 환경(건물)만 Sionna RT 경면+회절, 미세형상은 확산 미미 이유로 폐기. 표적 산란모델 없음
- **검출체인?** 레이더 검출 없음 — 'RCS/radar/CFAR' 문자열이 논문에 전무
- **우리와의 관계**: 다른 AI 이미지가 'Sionna 차용 ISAC'으로 든 예시지만, 실제로는 통신+온보드 다중센서 플랫폼. 우리 패시브 레이더 조각(RCS/검출) 없음

> **왜 이 둘도 싣나.** CISSIR 은 **NVIDIA 공식 'Made with Sionna' 쇼케이스의 유일한 ISAC 등재작**인데도 물리 표적 RCS 를 다루지 않는다 — 소형표적 RCS ISAC 이 스톡 Sionna 의 표준 용례가 아니라는 방증이다. SimART 는 다른 AI 답변이 'Sionna ISAC'으로 든 예지만, 실제 '센싱'은 카메라(YOLOv8)+GPS 라 우리 패시브 레이더 조각과 겹치지 않는다(정직한 구분).

---
## §4b. 신뢰성 등급 — 무엇을 얼마나 믿을까 (사용자 요청)

선행이라고 다 같은 무게가 아니다. **연구진·게재처**로 참고 가중을 나눈다:

> 신뢰성 순(참고 가중): peer-reviewed[Deterministic-Modeling EuCAP2026, Ziganshin] > 권위원천[Sionna-RT 창설(NVIDIA), CISSIR NVIDIA공식] > 대규모 preprint[LAMBDA 2.04TB, Great-X] > 일반 preprint[Temporal-GNN, SimART]. arXiv preprint 는 게재 확정 아님 — 방법론 참고엔 좋으나 수치는 보수적으로.

요컨대 **방법론**(어떻게 RCS 를 주입하나)은 preprint 라도 배울 게 많지만, **정량 수치**(σ 절대값 등)는 peer-reviewed(EuCAP)·권위원천(NVIDIA)부터 신뢰하고 preprint 는 보수적으로 읽는다.

---
## §5. 우리 주장은 선행에 의해 **지지**된다

### 📄단일출처  Sionna RT: Differentiable Ray Tracing for Radio Propagation Modeling
- **저자·출처**: Hoydis et al. (NVIDIA) — arXiv:2303.11103 (2023, v0.14~) — Sionna RT 창설 논문  ([1](https://arxiv.org/abs/2303.11103))
- **신뢰성(연구진·게재처)**: ⭐ **Sionna 창설 논문(NVIDIA, Hoydis)** — 엔진 저자 = 최상위 권위
- **무엇을 센싱?** n/a (엔진 논문)
- **표적 산란 처리**: 솔버 아키텍처를 문서화: 경면(이미지법 + Fibonacci 격자 광선탐색)·확산(계수 파라미터)·1차 회절. **경로별 복소이득을 반환하지, 표적 표면 위 SBR 장 적분이 아님**
- **검출체인?** n/a — ISAC 은 동기부여 주제로만 언급
- **우리와의 관계**: ⭐ **우리 주장의 1차 근거** — 스톡 솔버가 per-path gain 만 낸다는 것을 창설 논문이 문서화

**우리 report06 주장은 선행에 의해 지지됨: Deterministic-Modeling(EuCAP)이 'λ/10 메싱 불가·경면만이면 산란 과소평가'를 명시, Sionna RT 창설논문이 'per-path gain 반환'을 문서화.**

즉 report06 이 다섯 방식으로 실증한 'PathSolver 에 산란적분 없음'은 우리만의 발견이 아니라, **Sionna 창설 논문이 문서화**하고 **EuCAP 2026 논문이 재확인**한 사실이다. 우리가 한 일은 그 한계를 인정하고 **SBR+PO 를 부분적으로 더한 것**(report07)이다.

In [ ]:
# 이 리포트가 인용한 선행 논문 — 전부 prior_work.json 에서 (손으로 안 적음)
import json, os
J = json.load(open('outputs/prior_work.json', encoding='utf-8'))
for p in J['papers']:
    print(f"[{p['grade']:9s}] {p['venue']}")
    print(f"           {p['title'][:70]}")

---
## §6. 정리

1. **Sionna-ISAC 선행은 많다** — 드론(LAMBDA·Great-X)·차량(Deterministic-Modeling·Ziganshin)·추적(Temporal-GNN)·NVIDIA 공식(CISSIR)까지.
2. **소형표적 코히어런트 RCS-from-mesh 를 스톡 Sionna 로 푼 선행은 **없다**. 표준 우회 3가지: (b)확산계수 S[Great-X·Det-Modeling] (c)RCS 점표적 주입 h=h_bg+h_target[NIST 5GNRad·3GPP·MATLAB] (d)커스텀 산란 add-on[Ziganshin UTD, 우리 SBR+PO].**
3. **⭐ 우리 '자작 RCS(SBR+PO) → Sionna 채널 주입' 하이브리드는 **최신 Sionna-ISAC 선도 연구가 쓰는 바로 그 방식**이다: LAMBDA(2607.03826)=Sionna+CADFEKO UAV RCS, Temporal-GNN(2604.08306)=Sionna+점산란체 RCS 주입, clutter-aware ISAC=산란체별 통계RCS 부여. 차이는 RCS 를 상용툴(CADFEKO)이 아니라 우리가 SBR+PO 로 계산하고, 능동 FMCW 가 아니라 패시브 바이스태틱 OFDM 이라는 것 — 오히려 노벨티가 여기서 생긴다.**
4. 우리 위치와 '덜 점유된 틈새'는 다음 편에서 — **도구 지도(pw02)** 와 **포지셔닝(pw03)**.

> **다음** → [pw02 — 오픈소스 센싱/ISAC 도구 지도](pw02_opensource_tools.ipynb): NIST 5GNRad·RadarSimPy·OpenISAC 가 무엇을 해 주고, 우리 프로젝트에 무엇을 채택할까.